![alt text](../header.png)

## Running Vitis HLS from Jupyter Notebook

This cell executes a Vitis HLS synthesis flow directly from a Jupyter Notebook, allowing the automatic generation of an HLS project and its corresponding IP core using a TCL script.

The approach enables seamless integration of HLS-based hardware generation within a Python-based workflow, which is especially useful when working with tools such as hls4ml, automated design pipelines, or reproducible ML-to-FPGA flows.

In [1]:
# Import required modules
from pathlib import Path
import subprocess
import os
import shutil
import zipfile



In [2]:
# Configure the Vitis HLS environment
# This line prepends the Vitis HLS binary directory to the system PATH, ensuring that the vitis_hls executable can be called from within the notebook.

# THIS IS YOUR INSTALLATION PATH -> SHOULD BE MODIFIED ACCORDINGLY
os.environ['PATH'] = '/tools/Xilinx/Vitis_HLS/2024.1/bin:' + os.environ['PATH']

os.environ['PATH']



'/tools/Xilinx/Vitis_HLS/2024.1/bin:/home/ro/anaconda3/envs/neuralEnv/bin:/home/ro/google-cloud-sdk/bin:/home/ro/anaconda3/envs/neuralEnv/bin:/home/ro/anaconda3/condabin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/usr/games:/usr/local/games:/snap/bin:/snap/bin'

In [ ]:
# ------------------------------------------------------------------
# Define the HLS project directory
# ------------------------------------------------------------------
# PATH_HLS_PROJECT must point to the root directory of the HLS project
# generated by KalEdge or the corresponding notebook.
#
# This directory is created after extracting the HLS project archive
# downloaded from the KalEdge app.
# ------------------------------------------------------------------

PATH_HLS_PROJECT = '../hls4ml_prj_qat_qkeras_dma/hls4ml_prj_qat_qkeras/'

PROJ_DIR = Path(PATH_HLS_PROJECT)



In [ ]:
# Execute the HLS build script
res = subprocess.run(
    ["vitis_hls", "-f", "build_myproject_auto_accel.tcl"],
    cwd=PROJ_DIR,                      
    stdout=subprocess.PIPE, 
    stderr=subprocess.STDOUT,
    text=True,
)

# Display execution results
print("Return code:", res.returncode)
print(res.stdout)

In [ ]:
from helpers import parse_csynth_xml, hls_report_to_dataframe,csynth_xml_path, list_impl_ip, csynth_xml_from_project_root, export_hls_ip

path2  = Path(PATH_HLS_PROJECT) / 'hls4ml_prj_qat_qkeras_accel_prj'

report = parse_csynth_xml(csynth_xml_path(path2))
# print(report)
df = hls_report_to_dataframe(report)
print("HLS report:")
display(df)


HLS report:


,Used,Available,Util (%)
LUT,73667,70560,104.40
FF,4414,141120,3.13
DSP,54,360,15.00
BRAM_18K,2,432,0.46
URAM,0,0,NaN


In [ ]:

ip_dir, zips, dirs = list_impl_ip(path2)
print("impl/ip:", ip_dir)
print("zips:", [z.name for z in zips])
print("dirs:", [d.name for d in dirs])

print("csynth:", csynth_xml_from_project_root(path2))


impl/ip: ../hls4ml_prj_qap_qkeras_dma/hls4ml_prj_qap_qkeras/hls4ml_prj_qap_qkeras_accel_prj/solution1/impl/ip
zips: ['xilinx_com_hls_myproject_auto_accel_1_0.zip']
dirs: ['.Xil', 'constraints', 'doc', 'drivers', 'hdl', 'misc', 'xgui']
csynth: ../hls4ml_prj_qap_qkeras_dma/hls4ml_prj_qap_qkeras/hls4ml_prj_qap_qkeras_accel_prj/solution1/syn/report/csynth.xml


In [6]:

info = export_hls_ip(path2)

print("ZIP copied to:", info["zip"])
print("IP extracted at:", info["ip_dir"])

ZIP copied to: ip/hls4ml_prj_qap_qkeras_accel_prj/xilinx_com_hls_myproject_auto_accel_1_0.zip
IP extracted at: ip/hls4ml_prj_qap_qkeras_accel_prj


## Automated Vivado Execution via TCL Scripts

This notebook provides a utility function to execute Vivado TCL scripts in batch mode directly from Python.
It enables automation of synthesis and implementation flows by invoking Vivado programmatically, capturing console output, and handling execution errors.
This approach is particularly useful for reproducible FPGA workflows and integration within larger ML-to-hardware pipelines.

In [ ]:
def run_vivado_tcl(tcl_file_path, vivado_path="vivado"):

    """
    Executes a Vivado .tcl file for synthesis.

    tcl_file_path: Path to the .tcl file to be executed.
    vivado_path: Path to the Vivado executable (default assumes it's in PATH).
    """

    # ---- User should specify Vivado installation path
    # THIS IS YOUR INSTALLATION PATH -> SHOULD BE MODIFIED ACCORDINGLY
    vivado_path = '/tools/Xilinx/Vivado/2024.1/bin/vivado'
    # ----
    
    try:
        # Command to execute Vivado in batch mode
        command = [vivado_path, "-mode", "batch", "-source", tcl_file_path]

        # Run the command and capture output
        result = subprocess.run(
            command, 
            stdout=subprocess.PIPE, 
            stderr=subprocess.PIPE,     
            text=True
        )

        # Check if the command was successful
        if result.returncode == 0:
            print("Vivado .tcl script executed successfully!")
            print("Output:\n", result.stdout)
        else:
            print("Error during Vivado execution.")
            print("Error message:\n", result.stderr)

    except FileNotFoundError:
        print(f"Error: Vivado executable not found at '{vivado_path}'.")
    except Exception as e:
        print(f"Unexpected error: {e}")

In [ ]:

# ultra96-v2
tcl_file_path = 'tcl/gn-ultra-v96.tcl' 

run_vivado_tcl(tcl_file_path)